# MedAssist: fine-tuning LoRA no Google Colab
Ambiente: GPU T4 (Ambiente de execução > Alterar tipo > T4 GPU).

In [ ]:
!git clone <URL_DO_REPOSITORIO> medassist && cd medassist
%cd medassist
!pip install -q -r requirements-train.txt && pip install -q -e .

## 1. Preparar dados (sintéticos + opcionalmente PubMedQA)

In [ ]:
!wget -q https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json -O data/ori_pqal.json
!python -m medassist preparar --pubmedqa data/ori_pqal.json --limite 300

## 2. Baseline: modelo base sem adaptador

In [ ]:
!MEDASSIST_LLM=hf python -m medassist.finetune.evaluate --backend hf --no-adapter

## 3. Fine-tuning (QLoRA)

In [ ]:
!python -m medassist.finetune.train_lora --epochs 3 --out outputs/lora-medassist

## 4. Avaliação do modelo treinado

In [ ]:
!MEDASSIST_LLM=hf python -m medassist.finetune.evaluate --backend hf

## 5. Assistente e fluxo com o modelo treinado

In [ ]:
!MEDASSIST_LLM=hf python -m medassist perguntar "O que fazer na primeira hora da sepse?" -p 1
!MEDASSIST_LLM=hf python -m medassist fluxo -p 1 --aprovar

In [ ]:
# Curva de loss para o relatório
import json, matplotlib.pyplot as plt
h = json.load(open('outputs/lora-medassist/train_history.json'))
tr = [(x['step'], x['loss']) for x in h if 'loss' in x]
ev = [(x['step'], x['eval_loss']) for x in h if 'eval_loss' in x]
plt.plot(*zip(*tr), label='treino'); plt.plot(*zip(*ev), 'o-', label='validação'); plt.legend(); plt.xlabel('passo'); plt.ylabel('loss')
plt.savefig('docs/loss.png', dpi=150)